# Возможности MLflow

Этот ноутбук — обзор ключевых возможностей MLflow на примерах из наших экспериментов.

1. **Tracking API** — программный доступ к экспериментам и ранам
2. **Сравнение моделей** — поиск лучшего рана по метрикам
3. **Артефакты** — загрузка сохранённых файлов
4. **Model Registry** — версионирование и стадии моделей
5. **Автологирование** — автоматический трекинг без явных вызовов
6. **Загрузка модели для инференса** — из Registry по имени

UI доступен по адресу http://localhost:5001

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

mlflow.set_tracking_uri("http://localhost:5001")
client = MlflowClient()

## 1. Tracking API — обзор экспериментов и ранов

MLflow хранит всё в структуре: **Experiment** → **Run** → params, metrics, artifacts.

In [ ]:
# Список всех экспериментов
experiments = client.search_experiments()
for exp in experiments:
    print(f"  [{exp.experiment_id}] {exp.name} id={exp.experiment_id}")

In [ ]:
# Все раны эксперимента als_online
runs = client.search_runs(
    experiment_ids=[1],
    order_by=["metrics.map_at_10 DESC"],
)

runs_df = pd.DataFrame([
    {
        "run_name": r.info.run_name,
        "run_id": r.info.run_id[:8],
        "factors": r.data.params.get("factors"),
        "regularization": r.data.params.get("regularization"),
        "precision@10": r.data.metrics.get("precision_at_10"),
        "map@10": r.data.metrics.get("map_at_10"),
    }
    for r in runs
])
runs_df

## 2. Сравнение моделей — поиск лучшего рана

In [ ]:
# Поиск лучшего рана по метрике
best_run = client.search_runs(
    experiment_ids=[1],
    order_by=["metrics.map_at_10 DESC"],
    max_results=1,
)[0]

print(f"Лучший ран: {best_run.info.run_name}")
print(f"Run ID:     {best_run.info.run_id}")
print(f"Параметры:  {best_run.data.params}")
print(f"Метрики:    {best_run.data.metrics}")

In [ ]:
# Фильтрация ранов по параметрам
# Например: все раны с factors > 64
filtered = client.search_runs(
    experiment_ids=[1],
    filter_string="params.factors = '128'",
)
print(f"Раны с factors > 64: {len(filtered)}")
for r in filtered:
    print(f"  {r.info.run_name}: factors={r.data.params['factors']}, map@10={r.data.metrics.get('map_at_10', 'N/A')}")

In [ ]:
# Фильтрация по метрикам
# Все раны с AUC > 0.8 (из эксперимента reranker)
good_runs = client.search_runs(
    experiment_ids=[2],
    filter_string="metrics.val_auc > 0.8",
)
print(f"Раны с val_auc > 0.8: {len(good_runs)}")
for r in good_runs:
    print(f"  {r.info.run_name}: AUC={r.data.metrics['val_auc']:.4f}")

## 3. Артефакты — что сохранено в ране

In [ ]:
# Список артефактов лучшего ALS рана
artifacts = client.list_artifacts(best_run.info.run_id)
print(f"Артефакты рана {best_run.info.run_name}:")
for a in artifacts:
    print(f"  {a.path} ({a.file_size} bytes)" if a.file_size else f"  {a.path}/")

In [ ]:
# Скачиваем артефакт — маппинг user_id -> index
import json

local_path = client.download_artifacts(best_run.info.run_id, "user_id_to_idx.json", "/tmp")
with open(local_path) as f:
    mapping = json.load(f)
print(f"Загружен маппинг: {len(mapping)} пользователей")
print(f"Пример: {dict(list(mapping.items())[:3])}")

## 4. Model Registry — версионирование моделей

Model Registry позволяет:
- Зарегистрировать модель из рана
- Управлять версиями
- Назначать алиасы: `champion`, `challenger` и т.д.
- Загружать модель по имени: `models:/reranker@champion`

In [ ]:
# Регистрируем лучшую ALS модель
als_model_uri = f"mlflow-artifacts:/{best_run.info.experiment_id}/{best_run.info.run_id}/artifacts/als_model.pkl"

result = mlflow.register_model(
    model_uri=als_model_uri,
    name="als-recommender",
)
print(f"Зарегистрирована модель: {result.name}, версия: {result.version}")

In [ ]:
# Регистрируем CatBoost reranker
reranker_runs = client.search_runs(
    experiment_ids=[2],
    order_by=["metrics.val_auc DESC"],
    max_results=1,
)
reranker_run = reranker_runs[0]

result = mlflow.register_model(
    model_uri=f"runs:/{reranker_run.info.run_id}/reranker_model",
    name="catboost-reranker",
)
print(f"Зарегистрирована модель: {result.name}, версия: {result.version}")

In [ ]:
# Назначаем алиас champion — это текущая продовая модель
client.set_registered_model_alias("catboost-reranker", "prod", result.version)
print(f"Алиас 'prod' назначен на версию {result.version}")

# Список всех зарегистрированных моделей
for rm in client.search_registered_models():
    latest = rm.latest_versions
    for v in latest:
        aliases = client.get_model_version_by_alias(rm.name, "champion") if "champion" in [a for mv in latest for a in mv.aliases] else None
        print(f"  {rm.name} v{v.version} (aliases: {v.aliases})")

## 5. Автологирование

`mlflow.autolog()` автоматически логирует параметры, метрики и модель для популярных фреймворков (sklearn, catboost, lightgbm и др.).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Включаем автологирование
mlflow.autolog()

# Генерируем игрушечные данные
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

mlflow.set_experiment("autolog_demo")

# Просто вызываем .fit() — MLflow сам залогирует всё
rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

print("Модель обучена. Посмотрите в MLflow UI — ран залогирован автоматически!")
print("Залогированы: параметры (n_estimators, max_depth, ...), метрики (accuracy, f1, ...), модель")

In [ ]:
# Проверяем что залогировалось
autolog_runs = client.search_runs(experiment_ids=[3], max_results=1)
if autolog_runs:
    r = autolog_runs[0]
    print("Автоматически залогированные параметры:")
    for k, v in sorted(r.data.params.items()):
        print(f"  {k}: {v}")
    print(f"\nАвтоматически залогированные метрики:")
    for k, v in sorted(r.data.metrics.items()):
        print(f"  {k}: {v:.4f}")

mlflow.autolog(disable=True)  # Выключаем, чтобы не мешать дальше

## 6. Загрузка модели из Registry

В продакшене модель загружается не по `run_id`, а по имени и алиасу.  
Это позволяет обновлять модель без изменения кода сервиса.

In [ ]:
# Загружаем champion-модель по имени
model = mlflow.catboost.load_model("models:/catboost-reranker@prod")
print(f"Загружена модель: {type(model).__name__}")
print(f"Деревьев: {model.tree_count_}")
print(f"Фичей: {model.feature_names_}")

In [ ]:
# Также можно загрузить конкретную версию
model_v1 = mlflow.catboost.load_model("models:/catboost-reranker/1")
print(f"Модель версии 1: {model_v1.tree_count_} деревьев")